In [4]:
#Imports
import pickle
import numpy as np
import os

from tensorflow.keras.models import load_model # type: ignore
from tensorflow.keras.preprocessing.sequence import pad_sequences # type: ignore

#caminhos 
path_models = '../models/'
path_ia = path_models + 'chatbotIA.keras'
path_tokenizer = path_models + 'tokenizer.pkl'
path_label_encoder = path_models + 'label_encoder.pkl'

#parametro
maxlen = 20
limite = 0.65 #se tiver menos de 65% de certeza, não arrisca

#carregar modelo

model = load_model(path_ia)

# Carregar o tokenizer (o "dicionário" de palavras)
with open(path_tokenizer, 'rb') as f:
    tokenizer = pickle.load(f)

# Carregar o label encoder (o "tradutor" de IDs para respostas)
with open(path_label_encoder, 'rb') as f:
    label_encoder = pickle.load(f)



In [5]:
#Nova função de responder, usando o modelo ja treinado
def responder(mensagem): #função para carregar e abrir o arquivo

    print(f"Mensagem recebida: {mensagem}")

    # Pré-processar a mensagem do usuário, isso transforma a frase (ex: "oi") em uma sequência de IDs (ex: [2]) e finalmente posso deletar uma função feito para isso
    seq = tokenizer.texts_to_sequences([mensagem])

    if not seq or not seq[0]:
        return "Desculpe, não entendi. Tente usar outra palavra"

    # Padronizar a sequência (ex: [2]) em um vetor de tamanho 'max_len' (ex: [2, 0, 0, ...])
    padded_seq = pad_sequences(seq, maxlen=maxlen, padding='post')

    # Solicitar a previsão do modelo retorna um array de probabilidades (ex: [0.1, 0.05, 0.8, 0.05])
    predicao = model.predict(padded_seq)

    #Processo de decodificar 
    resposta_id = np.argmax(predicao)  # Tentar encontrar a melhor resposta np.argmax encontra o índice (ID) da maior probabilidade (ex: 2)
    confianca = predicao[0][resposta_id]
    resposta_texto = label_encoder.inverse_transform([resposta_id])[0] # "Destraduzir" a resposta, basicamente a label_encoder transforma o ID (ex: 2) de volta em texto (ex: "Desculpe, não entendi.")
    
    if confianca < limite:
        return "Poderia reformular a pergunta ?"
    
    # Retorna o texto da resposta, mas ainda esta tudo genérico e sem base de dados abrangente
    return resposta_texto


In [6]:
print("ChatBot: Olá! Digite 'sair' para encerrar.\n")

while True:
    usuario = input("Você: ")
    if usuario.lower() == "sair":
        print("ChatBot: Até mais!")
        break
    
    # Chamar nova função de IA, obs.: tem um ignore, pois vive apresentando alerta a qualquer modificação na função
    resposta = responder(usuario) # type: ignore
    print(f"ChatBot (IA): {resposta}", )


ChatBot: Olá! Digite 'sair' para encerrar.

Mensagem recebida: boa tarde
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
ChatBot (IA): Oi! Tudo bem?
Mensagem recebida: tudo sim
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
ChatBot (IA): Poderia reformular a pergunta ?
Mensagem recebida: tudo 
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
ChatBot (IA): Que bom! Como posso te ajudar?
Mensagem recebida: estou com duvida, como configura meu email no outlook ?
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
ChatBot (IA): Como configurar o email locaweb no outlook: https://www.locaweb.com.br/ajuda/wiki/configuracao-de-outlook-email-locaweb/ e também possui a opção: https://www.locaweb.com.br/ajuda/wiki/como-configurar-email-locaweb-no-office-365-email-locaweb
ChatBot: Até mais!
